In [ ]:
from pyspark.sql.functions import *
import json
from datetime import datetime
dbutils.widgets.text("env", "dev")
dbutils.widgets.text("jobRunId", "")
dbutils.widgets.text("taskRunId", "")
dbutils.widgets.text("workflowId", "")
env = dbutils.widgets.get("env").lower()
jobRunId = dbutils.widgets.get("jobRunId")
taskRunId = dbutils.widgets.get("taskRunId")
workflowId = dbutils.widgets.get("workflowId")

In [ ]:
# ---------------------------------------------------
# READ MASTER CONTROL (BRONZE CONFIG)
# ---------------------------------------------------
control = (
    spark.table("workspace.cfg_sch.master_control")
    .filter(f"workflowId = '{workflowId}' AND layer = 'BRONZE'")
    .first()
)

cfg = json.loads(control.colMapping)

source_path = control.sourceTable        # Volume path
target_table = control.targetTable
load_type = control.loadType
archive_flag = control.archive

columns = cfg["columns"]
bronze_col_list = [c["targetCol"] for c in columns]
archive_path = cfg.get("archivePath", None)
# ---------------------------------------------------
# READ FILES FROM VOLUME
# ---------------------------------------------------
df = (
    spark.read
    .option("header", True)
    .csv(source_path)
)
# ---------------------------------------------------
# ADD AUDIT COLUMNS
# ---------------------------------------------------
df = (
    df.withColumn("_ingestion_ts", current_timestamp())
      .withColumn("_job_run_id", lit(jobRunId))
      .withColumn("_source_path", lit(source_path))
)

# ---------------------------------------------------
# APPLY COLUMN MAPPING (rename + cast)
# ---------------------------------------------------
for c in columns:
    df = df.withColumnRenamed(c["sourceCol"], c["targetCol"])

df = df.select(bronze_col_list)

for c in df.columns:
    df = df.withColumn(
        c["targetCol"],
        col(c["targetCol"]).cast("string")
    )


# ---------------------------------------------------
# WRITE TO BRONZE BASED ON LOAD TYPE
# ---------------------------------------------------
if load_type.upper() == "APPEND":

    df.write.format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

elif load_type.upper() == "OVERWRITE":

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

else:
    raise Exception(f"Unsupported load type: {load_type}")

if archive_flag and archive_path:

    files = dbutils.fs.ls(source_path)

    # create timestamp once per batch
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    for f in files:
        print(f)
        source_file = f.path
        file_name = source_file.split("/")[-1]
        print(file_name)
        # add timestamp before extension
        if "." in file_name:
            name_part, ext_part = file_name.rsplit(".", 1)
            archived_name = f"{name_part}_{ts}.{ext_part}"
            dest_file = archive_path.rstrip("/") + "/" + archived_name
            dbutils.fs.mv(source_file, dest_file,)
        else:
            pass